# ImageNet Validation Script

The first thing we'll do is read in our saved model and logged metrics.

In [2]:
import torch
import torchvision

In [3]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(device)

cpu


In [4]:
def load_checkpoint(filename):
    checkpoint = torch.load(filename, map_location=device, weights_only=True)
    epochs = checkpoint['epochs']
    model_state_dict = checkpoint['model_state_dict']
    optimizer_state_dict = checkpoint['optimizer_state_dict']
    train_records = checkpoint['train_records']
    val_records = checkpoint['val_records']
    return (epochs, model_state_dict, optimizer_state_dict, 
            train_records, val_records)

In [5]:
epoch, model_state_dict, _, train_records, val_records = load_checkpoint("imagenet-checkpoint.pt")

In [6]:
model = torchvision.models.resnet50()
model.load_state_dict(model_state_dict)
model.eval();

In [7]:
# save model state dict
torch.save(model.state_dict(), "model-state-dict.pt")

Next we'll write a function to plot our training and validation losses and accuracies.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
def plot_metrics(
        epoch, train_records, val_records, 
        metrics=["Loss", "Top-1 accuracy", "Top-5 accuracy"],
    ):
    assert len(train_records) == len(val_records)
    n = len(train_records)
    fig, axs = plt.subplots(1, n, figsize=(3*n,3))
    x_train = np.linspace(0, epoch, len(train_records[0]), endpoint=False)
    x_val = np.linspace(0, epoch, len(val_records[0]), endpoint=True)
    for i, ax in enumerate(axs):
        ax.plot(x_train, train_records[i], label="Train")
        ax.plot(x_val, val_records[i], label="Val")
        ax.set_title(metrics[i])
        ax.set_ylim([0.0, None])
    fig.supxlabel("Iterations")
    axs[0].legend()
    fig.tight_layout()
    return fig, axs

In [ ]:
plot_metrics(epoch, train_records, val_records)
plt.show()

In [ ]:
print(f"Validation top-1 accuracy: {val_records[1][-1]}")
print(f"Validation top-5 accuracy: {val_records[2][-1]}")

# Test performance on validation dataset

In [ ]:
import sys
import os
sys.path.append(os.path.join(os.getcwd(), '../..'))
from utils.imagenet import get_val_transform

In [ ]:
nrows = ncols = 4
num_plot = nrows * ncols

In [ ]:
bucket_path = '/workspace/imagenet'
val_ds = torchvision.datasets.ImageFolder(
    bucket_path + "/val", transform=get_val_transform()
)
val_dl = torch.utils.data.DataLoader(
    val_ds, batch_size=num_plot, shuffle=True,
    num_workers=0, pin_memory=False, drop_last=False,
)

In [ ]:
imgs, labels = next(iter(val_dl))
preds = torch.argmax(model(imgs), dim=-1)

In [ ]:
print(imgs.shape)
print(labels)
print(preds)

In [ ]:
def plot_imgs(imgs, labels, preds, ):
    fig, axs = plt.subplots(nrows, ncols, figsize=(3*nrows, 3*ncols))
    for i, xax in enumerate(axs):
        for j, ax in enumerate(xax):
            k = i*nrows + j
            ax.imshow(imgs[k])
            ax.set_title(f"Label: {labels[k]}\nPredicted: {preds[k]}")
    fig.suptitle("Model predictions on validation set")
    fig.tight_layout()

In [ ]:
plot_imgs(imgs, labels, preds)
plt.show()